In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import requests
from functools import reduce
from sklearn.model_selection import train_test_split

# 0. Load Cleaned Dataset

In [ ]:
df_clean = pd.read_pickle('/content/drive/MyDrive/Thesis/df_clean.pkl')

df_clean.head()

In [ ]:
df_clean.info()

# 1. Data Integration

## 1.0 Load SES Data from Census API

In [ ]:
# import pandas as pd


# urls = {
#     "education_S1501": "https://api.census.gov/data/2019/acs/acs5/subject?get=group(S1501)&ucgid=pseudo(0500000US48113$1400000)",
#     "employment_S2301": "https://api.census.gov/data/2019/acs/acs5/subject?get=group(S2301)&ucgid=pseudo(0500000US48113$1400000)",
#     "poverty_S1701": "https://api.census.gov/data/2019/acs/acs5/subject?get=group(S1701)&ucgid=pseudo(0500000US48113$1400000)",
#     "income_S1901": "https://api.census.gov/data/2019/acs/acs5/subject?get=group(S1901)&ucgid=pseudo(0500000US48113$1400000)"
# }

# def load_census_data(url):
#     response = requests.get(url)
#     response.raise_for_status()

#     data = response.json()
#     columns = data[0]
#     rows = data[1:]

#     return pd.DataFrame(rows, columns=columns)

# ses_dfs = {}

# for name, url in urls.items():
#     df = load_census_data(url)

#     if "GEO_ID" in df.columns:
#         df["GEOID"] = df["GEO_ID"].str[-11:]

#     ses_dfs[name] = df

#     path = f"/content/drive/MyDrive/Thesis/{name}_raw_data.csv"
#     df.to_csv(path, index=False)

#     print(name, df.shape, "saved to:", path)

In [ ]:
base_path = "/content/drive/MyDrive/Thesis"

files = {
    "education_S1501": f"{base_path}/education_S1501_raw_data.csv",
    "employment_S2301": f"{base_path}/employment_S2301_raw_data.csv",
    "poverty_S1701": f"{base_path}/poverty_S1701_raw_data.csv",
    "income_S1901": f"{base_path}/income_S1901_raw_data.csv"
}

ses_dfs = {}

for name, path in files.items():
    df = pd.read_csv(path, dtype=str)

    if "GEOID" not in df.columns and "GEO_ID" in df.columns:
        df["GEOID"] = df["GEO_ID"].str[-11:]

    ses_dfs[name] = df

    print(name, df.shape)

In [ ]:
for name, df in ses_dfs.items():
    print(name)
    print("rows:", len(df))
    print("unique GEOID:", df["GEOID"].nunique())
    print(df["GEOID"].head())
    print()

In [ ]:
# # metadata

# import pandas as pd
# import requests

# metadata_urls = {
#     "education_S1501": "https://api.census.gov/data/2019/acs/acs5/subject/groups/S1501.json",
#     "employment_S2301": "https://api.census.gov/data/2019/acs/acs5/subject/groups/S2301.json",
#     "poverty_S1701": "https://api.census.gov/data/2019/acs/acs5/subject/groups/S1701.json",
#     "income_S1901": "https://api.census.gov/data/2019/acs/acs5/subject/groups/S1901.json"
# }

# def load_metadata(url):
#     response = requests.get(url)
#     response.raise_for_status()

#     data = response.json()
#     variables = data["variables"]

#     rows = []

#     for var_name, info in variables.items():
#         rows.append({
#             "variable": var_name,
#             "label": info.get("label"),
#             "concept": info.get("concept"),
#             "predicateType": info.get("predicateType"),
#             "group": info.get("group"),
#             "limit": info.get("limit"),
#             "attributes": info.get("attributes")
#         })

#     return pd.DataFrame(rows)

# metadata_dfs = {}

# for name, url in metadata_urls.items():
#     meta_df = load_metadata(url)

#     meta_df = meta_df.sort_values("variable").reset_index(drop=True)

#     metadata_dfs[name] = meta_df

#     path = f"/content/drive/MyDrive/Thesis/{name}_metadata_2019.csv"
#     meta_df.to_csv(path, index=False)

#     print(name, meta_df.shape, "saved to:", path)

In [ ]:
base_path = "/content/drive/MyDrive/Thesis"

metadata_files = {
    "education_S1501": f"{base_path}/education_S1501_metadata_2019.csv",
    "employment_S2301": f"{base_path}/employment_S2301_metadata_2019.csv",
    "poverty_S1701": f"{base_path}/poverty_S1701_metadata_2019.csv",
    "income_S1901": f"{base_path}/income_S1901_metadata_2019.csv"
}

metadata_dfs = {}

for name, path in metadata_files.items():
    meta_df = pd.read_csv(path, dtype=str)
    metadata_dfs[name] = meta_df

    print(name, meta_df.shape)

## 1.1 Create GEOID Column

In [ ]:
# Check current census tract values first
df_clean["Census_Tract_clean"].value_counts().head(20)

In [ ]:
# creat GEOID

df_clean["GEOID"] = df_clean["Census_Tract_clean"].where(
    df_clean["Census_Tract_clean"] != "UNKNOWN",
    pd.NA
)

df_clean["GEOID"] = "48113" + df_clean["GEOID"]

df_clean.loc[df_clean["Census_Tract_clean"] == "UNKNOWN", "GEOID"] = pd.NA

df_clean[["Census_Tract_clean", "GEOID"]].drop_duplicates().head(20)

In [ ]:
print(df_clean["GEOID"].isna().sum())
print(df_clean["GEOID"].dropna().head())
print(df_clean["GEOID"].dropna().str.startswith("48113").all())

## 1.2 Merge with GEOID

In [ ]:
# Make sure GEOID types match
df_clean["GEOID"] = df_clean["GEOID"].astype("string")

for name, df in ses_dfs.items():
    df["GEOID"] = df["GEOID"].astype("string").str.zfill(11)

In [ ]:
# Prepare SES dataframe

def prepare_ses_df(df, prefix):
    df = df.copy()
    df["GEOID"] = df["GEOID"].astype("string").str.zfill(11)

    # Drop repeated geographic identifier columns
    drop_cols = [col for col in ["GEO_ID", "ucgid"] if col in df.columns]
    df = df.drop(columns=drop_cols)

    # Prefix columns to avoid duplicate names
    rename_dict = {
        col: f"{prefix}_{col}"
        for col in df.columns
        if col != "GEOID"
    }

    df = df.rename(columns=rename_dict)
    return df

In [ ]:
education = prepare_ses_df(ses_dfs["education_S1501"], "edu")
employment = prepare_ses_df(ses_dfs["employment_S2301"], "emp")
poverty = prepare_ses_df(ses_dfs["poverty_S1701"], "pov")
income = prepare_ses_df(ses_dfs["income_S1901"], "inc")

In [ ]:
ses_merged = (
    education
    .merge(employment, on="GEOID", how="outer")
    .merge(poverty, on="GEOID", how="outer")
    .merge(income, on="GEOID", how="outer")
)

print(ses_merged.shape)
print(ses_merged["GEOID"].nunique())

In [ ]:
education = ses_dfs["education_S1501"][[
    "GEOID",
    "S1501_C02_015E"
]].rename(columns={
    "S1501_C02_015E": "bachelor_or_higher_pct"
})

employment = ses_dfs["employment_S2301"][[
    "GEOID",
    "S2301_C04_001E"
]].rename(columns={
    "S2301_C04_001E": "unemployment_rate"
})

poverty = ses_dfs["poverty_S1701"][[
    "GEOID",
    "S1701_C03_001E"
]].rename(columns={
    "S1701_C03_001E": "poverty_rate"
})

income = ses_dfs["income_S1901"][[
    "GEOID",
    "S1901_C01_012E"
]].rename(columns={
    "S1901_C01_012E": "median_household_income"
})

In [ ]:
# merge SES data

ses_selected = reduce(
    lambda left, right: left.merge(right, on="GEOID", how="outer"),
    [education, employment, poverty, income]
)

print(ses_selected.shape)
print(ses_selected["GEOID"].nunique())
print(ses_selected["GEOID"].duplicated().sum())

In [ ]:
ses_numeric_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

for col in ses_numeric_cols:
    ses_selected[col] = pd.to_numeric(ses_selected[col], errors="coerce")

In [ ]:
df_clean["GEOID"] = df_clean["GEOID"].astype("string")
ses_selected["GEOID"] = ses_selected["GEOID"].astype("string")

df_clean_ses = df_clean.merge(
    ses_selected,
    on="GEOID",
    how="left",
    validate="many_to_one"
)

print("Before merge:", df_clean.shape)
print("After merge:", df_clean_ses.shape)

https://www.census.gov/data/developers/data-sets/acs-1year/notes-on-acs-estimate-and-annotation-values.html

- missing code information
- Negative ACS special codes returned by the Census API were treated as missing values rather than valid numeric SES estimates.

In [ ]:
census_missing_codes = [
    -666666666,
    -222222222,
    -333333333,
    -444444444,
    -555555555,
    -888888888,
    -999999999
]

ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

for col in ses_cols:
    df_clean_ses[col] = pd.to_numeric(df_clean_ses[col], errors="coerce")
    df_clean_ses[col] = df_clean_ses[col].replace(census_missing_codes, np.nan)

In [ ]:
df_clean_ses["ses_matched"] = df_clean_ses[ses_cols].notna().any(axis=1)

## 1.3 Missing Check

In [ ]:
has_geoid = df_clean_ses["GEOID"].notna()

print("Rows with GEOID:", has_geoid.sum())
print("Rows without GEOID:", df_clean_ses["GEOID"].isna().sum())

print("Missing SES among rows with GEOID:")
print(df_clean_ses.loc[has_geoid, "bachelor_or_higher_pct"].isna().sum())

print("Missing SES rate among rows with GEOID:")
print(df_clean_ses.loc[has_geoid, "bachelor_or_higher_pct"].isna().mean())

Check Unmatched GEOIDs

In [ ]:
unmatched_geoids = (
    df_clean_ses.loc[
        df_clean_ses["GEOID"].notna() & df_clean_ses["bachelor_or_higher_pct"].isna(),
        "GEOID"
    ]
    .value_counts()
)

unmatched_geoids.head(30)

In [ ]:
df_geoids = set(df_clean["GEOID"].dropna().astype(str).unique())
ses_geoids = set(ses_selected["GEOID"].dropna().astype(str).unique())

print("df_clean unique GEOIDs:", len(df_geoids))
print("SES unique GEOIDs:", len(ses_geoids))
print("Matched unique GEOIDs:", len(df_geoids & ses_geoids))
print("Unmatched unique GEOIDs:", len(df_geoids - ses_geoids))

list(df_geoids - ses_geoids)[:30]

In [ ]:
print(df_clean["GEOID"].dropna().str.len().value_counts())
print(ses_selected["GEOID"].dropna().str.len().value_counts())

print(df_clean["GEOID"].dropna().head())
print(ses_selected["GEOID"].dropna().head())

In [ ]:
df_clean_ses["ses_matched"] = df_clean_ses["bachelor_or_higher_pct"].notna()

df_clean_ses["ses_matched"].value_counts(dropna=False)
df_clean_ses["ses_matched"].value_counts(normalize=True, dropna=False)

In [ ]:
df_clean_ses.groupby("ses_matched")["LOS_days"].describe()

## 1.4 2015 Missing ZIP to Census Tract Matching

In [ ]:
# 1. Define SES columns

ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

linkage_check_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate"
]

In [ ]:
# 2. Create direct SES linkage indicator

# 1 = ACS tract-level SES information was successfully linked
# 0 = no direct tract-level SES linkage

df_clean_ses["ses_matched"] = (
    df_clean_ses[linkage_check_cols].notna().any(axis=1)
).astype(int)

print("Direct SES linkage status:")
print(df_clean_ses["ses_matched"].value_counts(dropna=False))
print(df_clean_ses["ses_matched"].value_counts(normalize=True, dropna=False))

In [ ]:
# Load Crosswalk file

# HUD ZIP-TRACT crosswalk file
crosswalk_path = "/content/drive/MyDrive/Thesis/ZIP_TRACT_122015.xlsx"

zip_tract = pd.read_excel(crosswalk_path, dtype=str)
zip_tract.columns = zip_tract.columns.str.upper().str.strip()

print(zip_tract.head())
print(zip_tract.columns)

In [ ]:
# 4. Standardize ZIP, TRACT, and ratio columns

zip_tract["ZIP"] = zip_tract["ZIP"].astype(str).str.zfill(5)
zip_tract["TRACT"] = zip_tract["TRACT"].astype(str).str.zfill(11)

for col in ["RES_RATIO", "BUS_RATIO", "OTH_RATIO", "TOT_RATIO"]:
    zip_tract[col] = pd.to_numeric(zip_tract[col], errors="coerce")

# Dallas County FIPS prefix = 48113
zip_tract_dallas = zip_tract[
    zip_tract["TRACT"].str.startswith("48113")
].copy()

print("Dallas ZIP-TRACT crosswalk shape:", zip_tract_dallas.shape)
display(zip_tract_dallas.head())

In [ ]:
# 5. Build ACS SES lookup from original ACS table

# IMPORTANT:
# Use ses_selected, not df_clean_ses.
# df_clean_ses already contains failed merges, so it should not be used
# as the ACS lookup table.

acs_ses_lookup = ses_selected[["GEOID"] + ses_cols].copy()

acs_ses_lookup["GEOID"] = (
    acs_ses_lookup["GEOID"]
    .astype("string")
    .str.zfill(11)
)

for col in ses_cols:
    acs_ses_lookup[col] = pd.to_numeric(acs_ses_lookup[col], errors="coerce")
    acs_ses_lookup[col] = acs_ses_lookup[col].replace(census_missing_codes, np.nan)

print("ACS SES lookup shape:", acs_ses_lookup.shape)
print("Unique ACS GEOIDs:", acs_ses_lookup["GEOID"].nunique())

print("\nMissing values in ACS SES lookup:")
print(acs_ses_lookup[ses_cols].isna().sum())

display(acs_ses_lookup.head())

In [ ]:
# 6. Merge HUD ZIP-TRACT crosswalk with ACS SES data

zip_tract_ses = zip_tract_dallas.merge(
    acs_ses_lookup,
    left_on="TRACT",
    right_on="GEOID",
    how="left",
    validate="many_to_one"
)

print("ZIP-TRACT-SES shape:", zip_tract_ses.shape)

print("\nMissing SES values after merging ACS SES to ZIP-TRACT crosswalk:")
print(zip_tract_ses[ses_cols].isna().sum())

display(zip_tract_ses.head())

In [ ]:
# ------------------------------------------------------------
# 7. Create ZIP-level SES proxies using RES_RATIO-weighted averages
# ------------------------------------------------------------

def weighted_mean(group, value_col, weight_col="RES_RATIO"):
    valid = (
        group[value_col].notna()
        & group[weight_col].notna()
        & (group[weight_col] > 0)
    )

    if valid.sum() == 0:
        return np.nan

    weight_sum = group.loc[valid, weight_col].sum()

    if weight_sum == 0:
        return np.nan

    return np.average(
        group.loc[valid, value_col],
        weights=group.loc[valid, weight_col]
    )


zip_ses_rows = []

for zip_code, group in zip_tract_ses.groupby("ZIP"):
    row = {"ZIP": zip_code}

    # Diagnostic variables
    row["zip_res_ratio_sum_dallas"] = group["RES_RATIO"].sum()
    row["zip_tot_ratio_sum_dallas"] = group["TOT_RATIO"].sum()
    row["n_tracts_dallas"] = group["TRACT"].nunique()

    # RES_RATIO-weighted ACS SES proxy
    for col in ses_cols:
        row[col] = weighted_mean(group, col, "RES_RATIO")

    zip_ses_rows.append(row)

zip_ses = pd.DataFrame(zip_ses_rows)

print("ZIP-level SES proxy shape:", zip_ses.shape)

print("\nMissing values in ZIP-level SES proxy:")
print(zip_ses[ses_cols].isna().sum())

print("\nZIPs with RES_RATIO sum = 0:")
display(
    zip_ses[zip_ses["zip_res_ratio_sum_dallas"] == 0]
    [["ZIP", "zip_res_ratio_sum_dallas", "zip_tot_ratio_sum_dallas", "n_tracts_dallas"]]
    .head(20)
)

display(zip_ses.head())

In [ ]:
target_zips = [
    "75211", "75216", "75217", "75212",
    "75227", "75224", "75228", "75208",
    "75232", "75243", "75220", "75241"
]

display(
    zip_ses[zip_ses["ZIP"].isin(target_zips)]
    .sort_values("ZIP")
)

In [ ]:
# ------------------------------------------------------------
# 9. Recover ZIP-like values from incorrectly constructed GEOIDs
# ------------------------------------------------------------
# Example:
# GEOID 48113075211 -> suffix 075211 -> ZIP 75211

df_ses_fix = df_clean_ses.copy()

# Preserve direct tract-level SES linkage indicator
df_ses_fix["ses_direct_linked"] = df_ses_fix["ses_matched"]

# Convert GEOID to string safely
df_ses_fix["GEOID_str"] = df_ses_fix["GEOID"].astype("string")

# Extract suffix after Dallas County prefix 48113
df_ses_fix["geoid_suffix"] = pd.NA

mask_has_geoid = df_ses_fix["GEOID"].notna()

df_ses_fix.loc[mask_has_geoid, "geoid_suffix"] = (
    df_ses_fix.loc[mask_has_geoid, "GEOID_str"]
    .str.replace(r"^48113", "", regex=True)
)

# Convert bad tract-like suffix to ZIP-like value
df_ses_fix["zip_from_bad_geoid"] = (
    df_ses_fix["geoid_suffix"]
    .astype("string")
    .str.lstrip("0")
)

# Keep only Dallas ZIP-like values: 75xxx
df_ses_fix["zip_from_bad_geoid"] = np.where(
    df_ses_fix["zip_from_bad_geoid"].astype(str).str.match(r"^75\d{3}$", na=False),
    df_ses_fix["zip_from_bad_geoid"].astype(str),
    np.nan
)

print("ZIP-like values recovered from bad GEOIDs:")
print(df_ses_fix["zip_from_bad_geoid"].value_counts(dropna=False).head(30))

In [ ]:
# ------------------------------------------------------------
# 10. Merge ZIP-level SES proxies back to shelter records
# ------------------------------------------------------------

zip_ses_renamed = zip_ses.rename(columns={
    col: f"{col}_zip_weighted" for col in ses_cols
})

df_ses_fix = df_ses_fix.merge(
    zip_ses_renamed,
    left_on="zip_from_bad_geoid",
    right_on="ZIP",
    how="left"
)

print("Shape after ZIP SES merge:", df_ses_fix.shape)

print("\nPreview ZIP-weighted SES columns:")
display(
    df_ses_fix[
        ["zip_from_bad_geoid", "ZIP"] + [f"{col}_zip_weighted" for col in ses_cols]
    ].head()
)

In [ ]:
# Check rows where ZIP-like bad GEOID exists
display(
    df_ses_fix.loc[
        df_ses_fix["zip_from_bad_geoid"].notna(),
        ["zip_from_bad_geoid", "ZIP"] + [f"{col}_zip_weighted" for col in ses_cols]
    ].head(20)
)

In [ ]:
unmatched_zip_after_crosswalk = df_ses_fix.loc[
    df_ses_fix["zip_from_bad_geoid"].notna()
    & df_ses_fix["bachelor_or_higher_pct_zip_weighted"].isna(),
    "zip_from_bad_geoid"
].value_counts()

print(unmatched_zip_after_crosswalk)

In [ ]:
# ------------------------------------------------------------
# Fill only direct SES-unmatched records using ZIP-weighted SES
# ------------------------------------------------------------

df_ses_fix["ses_zip_crosswalked"] = 0

fill_mask = (
    (df_ses_fix["ses_direct_linked"] == 0) &
    df_ses_fix["zip_from_bad_geoid"].notna()
)

print("Candidate records for ZIP crosswalk fill:", fill_mask.sum())

for col in ses_cols:
    zip_col = f"{col}_zip_weighted"

    can_fill = (
        fill_mask &
        df_ses_fix[col].isna() &
        df_ses_fix[zip_col].notna()
    )

    print(f"{col} fill count:", can_fill.sum())

    df_ses_fix.loc[can_fill, col] = df_ses_fix.loc[can_fill, zip_col]
    df_ses_fix.loc[can_fill, "ses_zip_crosswalked"] = 1

print("\nZIP-crosswalked records:")
print(df_ses_fix["ses_zip_crosswalked"].value_counts(dropna=False))

print("\nRemaining missing SES values after ZIP crosswalk:")
print(df_ses_fix[ses_cols].isna().sum())

In [ ]:
print("Rows with zip_from_bad_geoid:")
print(df_ses_fix["zip_from_bad_geoid"].notna().sum())

print("\nRows with ZIP-weighted SES after merge:")
print(df_ses_fix["bachelor_or_higher_pct_zip_weighted"].notna().sum())

print("\nZIP match check:")
print(pd.crosstab(
    df_ses_fix["zip_from_bad_geoid"].notna(),
    df_ses_fix["bachelor_or_higher_pct_zip_weighted"].notna()
))

In [ ]:
# ============================================================
# View shelter ZIP-like Census_Tract values and matched HUD tracts
# ============================================================

# 1. Shelter records where bad GEOID was converted back to ZIP-like value
shelter_bad_zip_records = df_ses_fix.loc[
    df_ses_fix["zip_from_bad_geoid"].notna(),
    [
        "Census_Tract_clean",
        "GEOID",
        "zip_from_bad_geoid",
        "Intake_Year",
        "Animal_Type",
        "Intake_Type"
    ]
].copy()

print("Shelter records with ZIP-like values recovered from bad GEOID:")
print(shelter_bad_zip_records.shape)

display(shelter_bad_zip_records.head(20))

In [ ]:
# 2. Match shelter recovered ZIPs to HUD ZIP-TRACT crosswalk
shelter_zip_to_tract_view = shelter_bad_zip_records.merge(
    zip_tract_ses[
        [
            "ZIP",
            "TRACT",
            "RES_RATIO",
            "BUS_RATIO",
            "OTH_RATIO",
            "TOT_RATIO",
            "bachelor_or_higher_pct",
            "unemployment_rate",
            "poverty_rate",
            "median_household_income"
        ]
    ],
    left_on="zip_from_bad_geoid",
    right_on="ZIP",
    how="left"
)

print("Shelter ZIP-like records expanded to ZIP-TRACT matches:")
print(shelter_zip_to_tract_view.shape)

display(
    shelter_zip_to_tract_view[
        [
            "Census_Tract_clean",
            "GEOID",
            "zip_from_bad_geoid",
            "ZIP",
            "TRACT",
            "RES_RATIO",
            "bachelor_or_higher_pct",
            "unemployment_rate",
            "poverty_rate",
            "median_household_income",
            "Intake_Year",
            "Animal_Type",
            "Intake_Type"
        ]
    ].sort_values(["zip_from_bad_geoid", "RES_RATIO"], ascending=[True, False])
)

In [ ]:
# Shelter ZIP-like values that successfully matched to HUD ZIP-TRACT crosswalk
matched_shelter_zip_to_tract = shelter_zip_to_tract_view[
    shelter_zip_to_tract_view["ZIP"].notna()
].copy()

display(
    matched_shelter_zip_to_tract[
        [
            "Census_Tract_clean",
            "GEOID",
            "zip_from_bad_geoid",
            "ZIP",
            "TRACT",
            "RES_RATIO",
            "bachelor_or_higher_pct",
            "unemployment_rate",
            "poverty_rate",
            "median_household_income",
            "Intake_Year",
            "Animal_Type",
            "Intake_Type"
        ]
    ].sort_values(["zip_from_bad_geoid", "RES_RATIO"], ascending=[True, False])
)

In [ ]:
matched_zip_to_tract_unique = (
    matched_shelter_zip_to_tract[
        [
            "zip_from_bad_geoid",
            "ZIP",
            "TRACT",
            "RES_RATIO",
            "BUS_RATIO",
            "OTH_RATIO",
            "TOT_RATIO",
            "bachelor_or_higher_pct",
            "unemployment_rate",
            "poverty_rate",
            "median_household_income"
        ]
    ]
    .drop_duplicates()
    .sort_values(["zip_from_bad_geoid", "RES_RATIO"], ascending=[True, False])
)

display(matched_zip_to_tract_unique)

In [ ]:
display(
    zip_ses[
        [
            "ZIP",
            "zip_res_ratio_sum_dallas",
            "zip_tot_ratio_sum_dallas",
            "n_tracts_dallas",
            "bachelor_or_higher_pct",
            "unemployment_rate",
            "poverty_rate",
            "median_household_income"
        ]
    ].sort_values("ZIP")
)

In [ ]:
display(
    zip_ses[zip_ses["ZIP"] == "75006"]
)

In [ ]:
zip_code = "75006"

detail = zip_tract_ses[zip_tract_ses["ZIP"] == zip_code].copy()

for col in ses_cols:
    detail[f"{col}_weighted_component"] = detail[col] * detail["RES_RATIO"]

display(
    detail[
        [
            "ZIP",
            "TRACT",
            "RES_RATIO",
            "bachelor_or_higher_pct",
            "bachelor_or_higher_pct_weighted_component",
            "unemployment_rate",
            "unemployment_rate_weighted_component",
            "poverty_rate",
            "poverty_rate_weighted_component",
            "median_household_income",
            "median_household_income_weighted_component"
        ]
    ].sort_values("RES_RATIO", ascending=False)
)

print("Weighted average result for ZIP", zip_code)
for col in ses_cols:
    print(col, detail[f"{col}_weighted_component"].sum())

print("\nStored in zip_ses:")
display(zip_ses[zip_ses["ZIP"] == zip_code][["ZIP"] + ses_cols])


→ SES missing

→ SES missing

19,553 + 8,777 = 28,330

28,330 - 6,737 = 21,593

In [ ]:
ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

print("Missing SES values:")
print(df_ses_fix[ses_cols].isna().sum())

print("\nMissing SES percentage:")
print((df_ses_fix[ses_cols].isna().mean() * 100).round(2))

In [ ]:
# 2015 records only
df_2015 = df_ses_fix[df_ses_fix["Intake_Year"].astype(str) == "2015"].copy()

print("2015 rows:", df_2015.shape[0])

# SES missing count and percentage in 2015
ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

missing_2015 = pd.DataFrame({
    "missing_count": df_2015[ses_cols].isna().sum(),
    "missing_pct": df_2015[ses_cols].isna().mean() * 100
})

display(missing_2015.round(2))

In [ ]:
# Before crosswalk: df_clean_ses
before_missing = df_clean_ses[["bachelor_or_higher_pct", "unemployment_rate", "poverty_rate"]].isna().all(axis=1)

# After crosswalk: df_ses_fix
after_missing = df_ses_fix[["bachelor_or_higher_pct", "unemployment_rate", "poverty_rate"]].isna().all(axis=1)

print("Before ZIP crosswalk missing:", before_missing.sum())
print("After ZIP crosswalk missing:", after_missing.sum())
print("Recovered by ZIP crosswalk:", before_missing.sum() - after_missing.sum())

In [ ]:
df_ses_fix.to_pickle("/content/drive/MyDrive/Thesis/df_ses_fix.pkl")

## 1.5 SES feature EDA

- direct relationship weak, but retained for model comparison
   → ses_direct_linked = 1, ses_zip_crosswalked = 0

   → ses_direct_linked = 0, ses_zip_crosswalked = 1

   → ses_direct_linked = 0, ses_zip_crosswalked = 0

In [ ]:
indicator_cols = [
    "ses_matched",
    "ses_direct_linked",
    "ses_zip_crosswalked",
    "ses_available_after_crosswalk"
]

print([col for col in indicator_cols if col in df_ses_fix.columns])

In [ ]:
# Check merged data
df_ses_fix["ses_zip_crosswalked"].value_counts(dropna=False)
df_ses_fix["ses_zip_crosswalked"].value_counts(normalize=True, dropna=False)



In [ ]:
ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

df_ses_fix[ses_cols].isna().mean()

### 1.4.1 SES variable distribution

In [ ]:
df_ses_fix[ses_cols].describe()

In [ ]:
df_ses_fix["median_household_income"].describe()

In [ ]:
df_ses_fix[ses_cols].isna().mean()

In [ ]:
for col in ses_cols:
    print(col)
    print(df_ses_fix.loc[df_ses_fix[col] < 0, col].value_counts())
    print()

In [ ]:
ses_cols = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

# 1. Basic descriptive statistics
ses_summary = df_ses_fix[ses_cols].describe().T
ses_summary

In [ ]:
# 2. Missing rate, mean, median, skewness together
ses_distribution_summary = pd.DataFrame({
    "missing_count": df_ses_fix[ses_cols].isna().sum(),
    "missing_rate": df_ses_fix[ses_cols].isna().mean(),
    "mean": df_ses_fix[ses_cols].mean(),
    "median": df_ses_fix[ses_cols].median(),
    "std": df_ses_fix[ses_cols].std(),
    "min": df_ses_fix[ses_cols].min(),
    "max": df_ses_fix[ses_cols].max(),
    "skewness": df_ses_fix[ses_cols].skew()
})

ses_distribution_summary

In [ ]:
# 3. Histograms for each SES variable
for col in ses_cols:
    plt.figure(figsize=(7, 4))
    df_ses_fix[col].dropna().hist(bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

### 1.4.2 Correlation among SES variables


The SES variables showed expected correlations: median household income was negatively correlated with poverty rate and positively correlated with educational attainment. Because several SES indicators were moderately to strongly correlated, model interpretation, especially for linear models, was treated cautiously.

In [ ]:
df_ses_fix[ses_cols].corr()

### 1.4.3 Simple relationship with LOS / long_stay

- Simple correlations between SES indicators and LOS-related outcomes were close to zero, suggesting no strong direct linear association. SES variables were therefore treated as contextual predictors whose potential contribution would be evaluated through multivariate models rather than bivariate relationships alone.

In [ ]:
df_ses_fix[ses_cols + ["LOS_days"]].corr()["LOS_days"].sort_values()

In [ ]:
df_ses_fix[ses_cols + ["long_stay"]].corr()["long_stay"].sort_values()

SES variables showed expected correlations with each other, but their bivariate relationships with LOS and long-stay status were weak. Quartile-based comparisons also did not reveal a clear monotonic pattern in long-stay rates across SES levels. Therefore, SES variables were retained as contextual predictors for multivariate modeling rather than interpreted as having strong direct associations with LOS in the descriptive analysis.

In [ ]:
for col in ses_cols:
    q_col = col + "_quartile"
    df_ses_fix[q_col] = pd.qcut(
        df_ses_fix[col],
        q=4,
        labels=["Q1_low", "Q2", "Q3", "Q4_high"],
        duplicates="drop"
    )

    print("\n", q_col)
    print(df_ses_fix.groupby(q_col)["long_stay"].agg(["count", "mean"]))

# 2. Feature Engineering

## 2.2 Intake-related Features



### Intake_Subtype




In [ ]:
df_ses_fix['Intake_Subtype_clean'].value_counts()

In [ ]:
# Create feature-engineered intake subtype variable
df_ses_fix["Intake_Subtype_fe"] = df_ses_fix["Intake_Subtype_clean"].copy()

# Group clearly related subtypes
subtype_map = {
    # Return-related
    "RETURN30": "RETURN",

    # TNR-related
    "TRAP PROGRAM": "TNR",
    "TRAP NEUTER RETURN": "TNR",
    "FERAL FRIENDS": "TNR"
}

df_ses_fix["Intake_Subtype_fe"] = df_ses_fix["Intake_Subtype_fe"].replace(subtype_map)

# Keep major or interpretable categories
keep_subtypes = [
    "AT LARGE",
    "GENERAL",
    "CONFINED",
    "POSSIBLY OWNED",
    "QUARANTINE",
    "KEEP SAFE",
    "TREATMENT",
    "RETURN",
    "HEART WORM",
    "SURGERY",
    "CRUELTY",
    "EVICTION",
    "TNR",
    "APPOINT",
    "DANGEROUS"
]

# Group remaining small / ambiguous categories as OTHER
df_ses_fix["Intake_Subtype_fe"] = df_ses_fix["Intake_Subtype_fe"].where(
    df_ses_fix["Intake_Subtype_fe"].isin(keep_subtypes),
    "OTHER"
)

# Check result
df_ses_fix["Intake_Subtype_fe"].value_counts()

In [ ]:
# categories merged into OTHER
other_original_values = (
    df_ses_fix.loc[df_ses_fix["Intake_Subtype_fe"] == "OTHER", "Intake_Subtype_clean"]
    .value_counts()
)

other_original_values

### Intake_Condition

In [ ]:
df_ses_fix['Intake_Condition'].value_counts()

In [ ]:
df_ses_fix[
    df_ses_fix["Intake_Condition"].isin(["CRITICAL", "FATAL"])
][[
    "Intake_Condition",
    "Outcome_Type",
    "Intake_Type",
    "Intake_Subtype",
    "Animal_Type",
    "Intake_Date",
    "Outcome_Date",
    "LOS_days"
]]

In [ ]:
pd.crosstab(
    df_ses_fix["Intake_Condition"],
    df_ses_fix["Outcome_Type"],
    margins=True
).loc[["CRITICAL", "FATAL"]]

In [ ]:
df_ses_fix[
    df_ses_fix["Intake_Condition"].isin(["CRITICAL", "FATAL"])
].groupby("Intake_Condition")["LOS_days"].describe()

In [ ]:
df_ses_fix["Intake_Condition_fe"] = (
    df_ses_fix["Intake_Condition"]
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .replace({
        "CRITICAL": "CRITICAL_OR_FATAL",
        "FATAL": "CRITICAL_OR_FATAL"
    })
)

In [ ]:
df_ses_fix['Intake_Condition_fe'].value_counts()

### Chip_Status

In [ ]:
df_ses_fix['Chip_Status'].value_counts()

In [ ]:
df_ses_fix[df_ses_fix['Chip_Status'] == 'WILDLIFE - UNABEL TO SCAN']['Animal_Type'].value_counts()

In [ ]:
df_ses_fix["Chip_Status_fe"] = df_ses_fix["Chip_Status"].replace({
    "WILDLIFE - UNABEL TO SCAN": "UNABLE TO SCAN"
})

df_ses_fix["Chip_Status_fe"].value_counts()

### Animal_Origin

In [ ]:
df_ses_fix['Animal_Origin'].value_counts()

In [ ]:
df_ses_fix["Animal_Origin_fe"] = (
    df_ses_fix["Animal_Origin"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df_ses_fix["Animal_Origin_fe"] = df_ses_fix["Animal_Origin_fe"].replace({
    "OPS": "OTHER",
    "NIGHT DROP": "OTHER",
    "AGGOPS": "OTHER",
    "RAPID": "OTHER"
})

df_ses_fix["Animal_Origin_fe"].value_counts()

## 2.3 Time-related Features

In [ ]:
df_ses_fix['Intake_Month'] = df_ses_fix['Intake_Date'].dt.month

In [ ]:
df_ses_fix['Intake_Month'].value_counts()

## 2.4 Define Final Feature Sets

In [ ]:
df_ses_fix.info()

In [ ]:
categorical_features = [
    "Animal_Type",
    "Animal_Breed_clean",
    "Intake_Type",
    "Intake_Subtype_fe",
    "Intake_Condition_fe",
    "Chip_Status_fe",
    "Animal_Origin_fe",
    "Intake_Year",
    "Intake_Month"
]

numeric_features_no_ses = []

numeric_features_with_ses = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income",
    "ses_direct_linked",
    "ses_zip_crosswalked"
]



In [ ]:
df_ses_fix["Intake_Year"] = (
    df_ses_fix["Intake_Year"]
    .astype("Int64")
    .astype("string")
)

df_ses_fix["Intake_Month"] = (
    df_ses_fix["Intake_Month"]
    .astype("Int64")
    .astype("string")
)

In [ ]:
features_no_ses = categorical_features + numeric_features_no_ses
features_with_ses = categorical_features + numeric_features_with_ses

### 2.4.1 Create Dataset for modeling

In [ ]:
df_model_no_ses = df_ses_fix.copy()

df_model_ses = df_ses_fix.copy()

### 2.4.2 X,y for classification

In [ ]:
# Full sample, no SES
X_no_ses = df_ses_fix[features_no_ses].copy()
y_no_ses = df_ses_fix["long_stay"].copy()

# Full sample, with SES
X_ses = df_ses_fix[features_with_ses].copy()
y_ses = df_ses_fix["long_stay"].copy()

In [ ]:
print("No SES missing:")
print(X_no_ses.isna().mean().sort_values(ascending=False))

print("\nWith SES missing:")
print(X_ses.isna().mean().sort_values(ascending=False))

### 2.4.3 Train/Test  Split

In [ ]:
# Same full-sample train/test indices for both no-SES and with-SES models
train_idx, test_idx = train_test_split(
    df_ses_fix.index,
    test_size=0.2,
    random_state=42,
    stratify=df_ses_fix["long_stay"]
)

# Full sample, no SES
X_train_no_ses = X_no_ses.loc[train_idx].copy()
X_test_no_ses = X_no_ses.loc[test_idx].copy()
y_train_no_ses = y_no_ses.loc[train_idx].copy()
y_test_no_ses = y_no_ses.loc[test_idx].copy()

# Full sample, with SES
X_train_ses = X_ses.loc[train_idx].copy()
X_test_ses = X_ses.loc[test_idx].copy()
y_train_ses = y_ses.loc[train_idx].copy()
y_test_ses = y_ses.loc[test_idx].copy()

In [ ]:
print(X_train_no_ses.shape, X_train_ses.shape)
print(X_test_no_ses.shape, X_test_ses.shape)

print(X_train_no_ses.index.equals(X_train_ses.index))
print(X_test_no_ses.index.equals(X_test_ses.index))

print(y_train_no_ses.equals(y_train_ses))
print(y_test_no_ses.equals(y_test_ses))

1. train/test split: 80/20


## 2.5 Save Dataframe for modeling

In [ ]:
ses_features = [
    "bachelor_or_higher_pct",
    "unemployment_rate",
    "poverty_rate",
    "median_household_income"
]

print("X_train_no_ses:", X_train_no_ses.shape)
print("X_test_no_ses:", X_test_no_ses.shape)

print("X_train_ses:", X_train_ses.shape)
print("X_test_ses:", X_test_ses.shape)

print("y_train_no_ses:", y_train_no_ses.shape)
print("y_test_no_ses:", y_test_no_ses.shape)

print("y_train_ses:", y_train_ses.shape)
print("y_test_ses:", y_test_ses.shape)


print("\nSame train index:", X_train_no_ses.index.equals(X_train_ses.index))
print("Same test index:", X_test_no_ses.index.equals(X_test_ses.index))

print("Same train target:", y_train_no_ses.equals(y_train_ses))
print("Same test target:", y_test_no_ses.equals(y_test_ses))


print("\nNo SES train:")
print(y_train_no_ses.value_counts(normalize=True))

print("\nNo SES test:")
print(y_test_no_ses.value_counts(normalize=True))

print("\nWith SES train:")
print(y_train_ses.value_counts(normalize=True))

print("\nWith SES test:")
print(y_test_ses.value_counts(normalize=True))


print("\nSES missing values in train:")
print(X_train_ses[ses_features].isna().sum())

print("\nSES missing values in test:")
print(X_test_ses[ses_features].isna().sum())

In [ ]:
print("X_train_no_ses:", X_train_no_ses.shape)
print("X_test_no_ses:", X_test_no_ses.shape)
print("X_train_ses:", X_train_ses.shape)
print("X_test_ses:", X_test_ses.shape)

print("Same train index:", X_train_no_ses.index.equals(X_train_ses.index))
print("Same test index:", X_test_no_ses.index.equals(X_test_ses.index))

In [ ]:
split_data = {
    "features_no_ses": features_no_ses,
    "features_with_ses": features_with_ses,
    "categorical_features": categorical_features,
    "numeric_features_with_ses": numeric_features_with_ses,

    "X_train_no_ses": X_train_no_ses,
    "X_test_no_ses": X_test_no_ses,
    "y_train_no_ses": y_train_no_ses,
    "y_test_no_ses": y_test_no_ses,

    "X_train_ses": X_train_ses,
    "X_test_ses": X_test_ses,
    "y_train_ses": y_train_ses,
    "y_test_ses": y_test_ses
}

with open("/content/drive/MyDrive/Thesis/model_splits_classification_final.pkl", "wb") as f:
    pickle.dump(split_data, f)

df_ses_fix.to_pickle("/content/drive/MyDrive/Thesis/df_ses_fix.pkl")